# Minority-class study on the GTCN-G style model

The paper names class imbalance as the problem its residual connection is meant to solve. In the ablation notebook the full GTCN-G style model scored a weighted F1 of 0.9925 but a macro F1 of only 0.55, and Analysis, Backdoor and DoS stayed below 0.19. This notebook keeps the same model, data subset (flows of UNSW-NB15_1.csv) and 5:2:3 split, and changes only how the rare classes are treated:

| Loss | What it does |
|---|---|
| plain | cross-entropy, as in the ablation notebook |
| sqrt weights | cross-entropy weighted by the square root of the balanced class weights |
| balanced weights | cross-entropy weighted by the balanced class weights |
| logit-adjusted | cross-entropy on logits plus the log of the class frequency (Menon et al.), plain logits at test time |

Every trained model is then decoded twice: by the largest probability, and after multiplying each class probability by a scale tuned on the validation set for macro F1 (same procedure as the LightGBM notebook). All choices, including the stopping epoch (validation macro F1 here, not weighted F1), are made on validation; the test set is used once per model.

In [ ]:
import copy

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

DEBUG = False
SEEDS = [42] if DEBUG else [42, 43, 44]
RUN = ["plain", "sqrt weights", "balanced weights", "logit-adjusted"]
PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HEADS, HEAD_DIM = 6, 24
FANOUT = 8
K = 2 * FANOUT
BRANCH = 64
DROPOUT = 0.2
ATTN_DROPOUT = 0.5
LR = 0.001
BATCH = 500
MAX_EPOCHS = 3 if DEBUG else 60
PATIENCE = 3 if DEBUG else 10

## 1. Data and 5:2:3 split

In [ ]:
df = pd.concat([pd.read_parquet(f"{PATH}/train.parquet"), pd.read_parquet(f"{PATH}/test.parquet")])
df = df[df.index < 700_001]
graph = pd.concat([pd.read_csv(f"{PATH}/train_graph.csv", index_col="row_id"),
                   pd.read_csv(f"{PATH}/test_graph.csv", index_col="row_id")]).loc[df.index]
if DEBUG:
    df = df.sample(200_000, random_state=0)
    graph = graph.loc[df.index]

class_names = list(joblib.load(f"{PATH}/preprocessing.pkl")["label_encoder"].classes_)
n_classes = len(class_names)
feature_cols = [c for c in df.columns if c != "label"]

train_ids, rest_ids = train_test_split(df.index, train_size=0.5, stratify=df["label"], random_state=0)
val_ids, test_ids = train_test_split(rest_ids, train_size=0.4, stratify=df.loc[rest_ids, "label"], random_state=0)
print({k: len(v) for k, v in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}, "| features", len(feature_cols))

## 2. Graphs

Nodes are (IP, port) endpoints and flows are edges, so the line graph's nodes are flows. For every endpoint the flows that touch it are stored as a neighbour list (CSR); the neighbours of a flow are the flows at its two endpoints. Each split gets its own graph.

In [ ]:
def build_graph(ids):
    feat_df, graph_df = df.loc[ids], graph.loc[ids]
    n = len(feat_df)
    ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    node_id, uniq = pd.factorize(ip_code.astype("int64") * 70_000 + ports)
    node_id = torch.from_numpy(node_id).to(DEVICE)
    src, dst = node_id[:n], node_id[n:]

    nodes = torch.cat([src, dst])
    flow = torch.cat([torch.arange(n, device=DEVICE)] * 2)
    order = torch.argsort(nodes)
    ptr = torch.zeros(len(uniq) + 1, dtype=torch.long, device=DEVICE)
    ptr[1:] = torch.cumsum(torch.bincount(nodes, minlength=len(uniq)), 0)
    return {"src": src, "dst": dst, "eflow": flow[order], "ptr": ptr,
            "rid": torch.from_numpy(feat_df.index.to_numpy("int64")).to(DEVICE),
            "x": torch.from_numpy(feat_df[feature_cols].to_numpy("float32")).to(DEVICE),
            "y": torch.from_numpy(feat_df["label"].to_numpy("int64")).to(DEVICE)}


G = {"train": build_graph(train_ids), "val": build_graph(val_ids), "test": build_graph(test_ids)}
for name, g in G.items():
    deg = torch.diff(g["ptr"]).float()
    print(f"{name:<6} flows {len(g['src']):>9,} | endpoints {len(deg):>9,} | mean flows per endpoint {deg.mean():.2f}")

## 3. Model

For a batch of flows the model draws 8 neighbouring flows from each endpoint (16 in total, the flow itself excluded), orders them by time and feeds four branches: attention over the neighbours, a residual branch with the flow's own features, a gated temporal convolution over the ordered neighbours and a mean graph convolution. The branch outputs are concatenated and classified. A flow whose endpoints carry no other flow has no neighbours; its neighbour branches are zero and only the residual branch speaks.

In [ ]:
def neighbours(g, idx, mode):
    b = len(idx)
    if mode == "none":
        return torch.zeros(b, K, dtype=torch.long, device=DEVICE), torch.zeros(b, K, dtype=torch.bool, device=DEVICE)
    if mode == "shuffled":
        return torch.randint(0, len(g["src"]), (b, K), device=DEVICE), torch.ones(b, K, dtype=torch.bool, device=DEVICE)
    ends = torch.cat([g["src"][idx], g["dst"][idx]])
    start = g["ptr"][ends]
    deg = g["ptr"][ends + 1] - start
    pos = start[:, None] + (torch.rand(2 * b, FANOUT, device=DEVICE) * deg[:, None]).long()
    flows = g["eflow"][pos]
    flows = torch.cat([flows[:b], flows[b:]], 1)
    return flows, flows != idx[:, None]


class GTCNG(nn.Module):
    def __init__(self, n_feat, n_classes, residual=True, attn=True, temporal=True, spatial=True, graph="real"):
        super().__init__()
        self.residual, self.attn, self.temporal, self.spatial, self.graph = residual, attn, temporal, spatial, graph
        width = HEADS * HEAD_DIM
        self.proj = nn.Linear(n_feat, width, bias=False)
        self.a_nbr = nn.Parameter(torch.randn(HEADS, HEAD_DIM) * 0.1)
        self.a_self = nn.Parameter(torch.randn(HEADS, HEAD_DIM) * 0.1)
        self.attn_drop = nn.Dropout(ATTN_DROPOUT)
        if residual:
            self.res = nn.Linear(n_feat, BRANCH)
            width += BRANCH
        if temporal:
            self.t_in = nn.Linear(n_feat, BRANCH)
            self.t_conv = nn.ModuleList([nn.Conv1d(BRANCH, 2 * BRANCH, 3, dilation=d) for d in (1, 2)])
            width += BRANCH
        if spatial:
            self.sp = nn.Linear(n_feat, BRANCH)
            width += BRANCH
        self.head = nn.Sequential(nn.Linear(width, 128), nn.ReLU(), nn.Dropout(DROPOUT), nn.Linear(128, n_classes))

    def forward(self, g, idx):
        b = len(idx)
        x_self = g["x"][idx]
        flows, mask = neighbours(g, idx, self.graph)
        order = torch.where(mask, g["rid"][flows], torch.full_like(flows, -1)).argsort(1)
        flows, mask = flows.gather(1, order), mask.gather(1, order)
        m = mask.float()
        x_nbr = g["x"][flows] * m[..., None]
        count = m.sum(1).clamp(min=1)

        w_nbr = self.proj(x_nbr).view(b, K, HEADS, HEAD_DIM)
        if self.attn:
            w_self = self.proj(x_self).view(b, HEADS, HEAD_DIM)
            score = F.leaky_relu((w_nbr * self.a_nbr).sum(-1) + (w_self * self.a_self).sum(-1)[:, None])
            alpha = torch.softmax(score.masked_fill(~mask[..., None], -1e9), dim=1) * m[..., None]
            agg = (self.attn_drop(alpha)[..., None] * w_nbr).sum(1)
        else:
            agg = (w_nbr * m[..., None, None]).sum(1) / count[:, None, None]
        parts = [F.elu(agg).flatten(1)]

        if self.residual:
            parts.append(F.relu(self.res(x_self)))
        if self.temporal:
            h = (self.t_in(x_nbr) * m[..., None]).transpose(1, 2)
            for conv, dilation in zip(self.t_conv, (1, 2)):
                gate, value = conv(F.pad(h, (2 * dilation, 0))).chunk(2, 1)
                h = torch.tanh(value) * torch.sigmoid(gate) + h
            parts.append(h[:, :, -1])
        if self.spatial:
            parts.append(F.relu(self.sp(x_nbr.sum(1) / count[:, None])))
        return self.head(torch.cat(parts, 1))

In [ ]:
@torch.no_grad()
def predict(model, g, batch=5000):
    model.eval()
    return torch.cat([model(g, torch.arange(i, min(i + batch, len(g["src"])), device=DEVICE))
                      for i in range(0, len(g["src"]), batch)])


def scores(y_true, pred):
    per_class = f1_score(y_true, pred, labels=np.arange(n_classes), average=None, zero_division=0)
    return {"weighted F1": f1_score(y_true, pred, average="weighted", zero_division=0),
            "macro F1": per_class.mean(), "accuracy": accuracy_score(y_true, pred)}, per_class


def tune_scales(proba, y_true, passes=2, grid=np.geomspace(0.25, 4, 13)):
    scale = np.ones(n_classes)
    for _ in range(passes):
        for c in range(n_classes):
            trial = []
            for m in grid:
                s = scale.copy()
                s[c] = m
                trial.append(f1_score(y_true, (proba * s).argmax(1), average="macro", zero_division=0))
            scale[c] = grid[int(np.argmax(trial))]
    return scale


counts = torch.bincount(G["train"]["y"], minlength=n_classes).float()
balanced = counts.sum() / (n_classes * counts)
LOSSES = {
    "plain": (None, 0.0),
    "sqrt weights": (balanced.sqrt(), 0.0),
    "balanced weights": (balanced, 0.0),
    "logit-adjusted": (None, 1.0),
}
log_prior = torch.log(counts / counts.sum())

## 4. Training

Batches of 500, learning rate 0.001 halved after 5 epochs without improvement, early stopping after 10 epochs without improvement of validation macro F1, at most 60 epochs.

In [ ]:
def run(loss_name, seed):
    torch.manual_seed(seed)
    weight, tau = LOSSES[loss_name]
    train_loss = nn.CrossEntropyLoss(weight=None if weight is None else weight.to(DEVICE))
    shift = (tau * log_prior).to(DEVICE)
    model = GTCNG(len(feature_cols), n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)
    g_train = G["train"]
    y_val, y_test = G["val"]["y"].cpu().numpy(), G["test"]["y"].cpu().numpy()
    n_train = len(g_train["src"])

    history, best, bad = [], {"f1": -1.0}, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=DEVICE)
        total = torch.zeros((), device=DEVICE)
        for i in range(0, n_train, BATCH):
            idx = perm[i:i + BATCH]
            loss = train_loss(model(g_train, idx) + shift, g_train["y"][idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.detach() * len(idx)

        val_logits = predict(model, G["val"])
        val, _ = scores(y_val, val_logits.argmax(1).cpu().numpy())
        history.append({"loss": loss_name, "seed": seed, "epoch": epoch, "train loss": total.item() / n_train,
                        "val loss": F.cross_entropy(val_logits, G["val"]["y"]).item(),
                        "val weighted F1": val["weighted F1"], "val macro F1": val["macro F1"]})
        print(f"{loss_name} | seed {seed} epoch {epoch:>3} | train loss {history[-1]['train loss']:.4f} "
              f"| val weighted F1 {val['weighted F1']:.4f} | val macro F1 {val['macro F1']:.4f}", flush=True)

        sched.step(val["macro F1"])
        if val["macro F1"] > best["f1"]:
            best, bad = {"f1": val["macro F1"], "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best["state"])
    val_proba = torch.softmax(predict(model, G["val"]), 1).cpu().numpy()
    test_proba = torch.softmax(predict(model, G["test"]), 1).cpu().numpy()
    scale = tune_scales(val_proba, y_val)
    out = {}
    for decoding, pred in [("largest probability", test_proba.argmax(1)), ("tuned class scales", (test_proba * scale).argmax(1))]:
        s, per_class = scores(y_test, pred)
        out[decoding] = ({"loss": loss_name, "decoding": decoding, "seed": seed, "best epoch": best["epoch"], "last epoch": epoch, **s},
                         {"loss": loss_name, "decoding": decoding, "seed": seed, **dict(zip(class_names, per_class))}, pred.astype("int8"))
    print("tuned scales:", dict(zip(class_names, scale.round(2))), flush=True)
    return pd.DataFrame(history), out


histories, rows, class_rows, preds = [], [], [], {}
for loss_name in RUN:
    for seed in SEEDS:
        history, out = run(loss_name, seed)
        histories.append(history)
        for decoding, (row, class_row, pred) in out.items():
            rows.append(row)
            class_rows.append(class_row)
            preds[f"{loss_name}|{decoding}|{seed}"] = pred
        pd.concat(histories).to_csv("/kaggle/working/minority_history.csv", index=False)
        pd.DataFrame(rows).to_csv("/kaggle/working/minority_runs.csv", index=False)
        pd.DataFrame(class_rows).to_csv("/kaggle/working/minority_per_class.csv", index=False)
history = pd.concat(histories)
results = pd.DataFrame(rows)
class_results = pd.DataFrame(class_rows)
pd.DataFrame(preds, index=test_ids).to_parquet("/kaggle/working/minority_test_pred.parquet")
np.save("/kaggle/working/minority_test_y.npy", G["test"]["y"].cpu().numpy())

## 5. Test results

In [ ]:
metrics = ["weighted F1", "macro F1", "accuracy"]
summary = results.groupby(["loss", "decoding"], sort=False)[metrics].agg(["mean", "std"])
print(summary.round(4).to_string())
summary.to_csv("/kaggle/working/minority_summary.csv")
print(class_results.drop(columns="seed").groupby(["loss", "decoding"], sort=False).mean().round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col in zip(axes, ["weighted F1", "macro F1"]):
    sns.barplot(data=results, y="loss", x=col, hue="decoding", errorbar="sd", ax=ax)
    ax.set_title(f"Test {col}")
axes[0].set_xlim(0.95, 1.0)
plt.tight_layout()
plt.savefig("/kaggle/working/fig_01.png", dpi=130, bbox_inches="tight")
plt.show()

per_class = class_results.drop(columns="seed").groupby(["loss", "decoding"], sort=False).mean()
fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(per_class, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
ax.set_title("Test F1 per class (mean over seeds)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_02.png", dpi=130, bbox_inches="tight")
plt.show()